In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

In [20]:
df = pd.read_csv('data/joined_with_tt_scores.csv')
df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])

In [21]:
#average rating per user
user_avg_rating = df.groupby('userId')['rating'].mean()

#average rating for user per genre
genre_cols = ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

# Step 1: Multiply each genre column by rating
weighted = df[genre_cols].multiply(df['rating'], axis=0)

# Step 2: Sum ratings per user per genre
genre_sum = weighted.groupby(df['userId']).sum()

# Step 3: Count how many ratings per genre per user
genre_count = df[genre_cols].groupby(df['userId']).sum()

# Step 4: Compute averages
genre_avg = genre_sum / genre_count

user_features = genre_avg.reset_index()

user_features = user_features.rename(
    columns={col: f"user_avg_{col}" for col in genre_cols}
)

In [22]:
# Movie average rating
movie_sum = df.groupby('movieId')['rating'].transform('sum')
movie_count = df.groupby('movieId')['rating'].transform('count')

df['movie_avg_loo'] = (movie_sum - df['rating']) / (movie_count - 1)

# Handle edge case (movies with only 1 rating)
global_avg = df['rating'].mean()
df['movie_avg_loo'] = df['movie_avg_loo'].fillna(global_avg)

In [ ]:
#Merge dataframes
df = df.merge(user_features, on='userId', how='left')

Index(['userId', 'movieId', 'title', 'rating', 'rating_timestamp', 'tag',
       'tag_timestamp', '(no genres listed)', 'Action', 'Adventure',
       'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery',
       'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'two_tower_score',
       'movie_avg_loo', 'user_avg_(no genres listed)', 'user_avg_Action',
       'user_avg_Adventure', 'user_avg_Animation', 'user_avg_Children',
       'user_avg_Comedy', 'user_avg_Crime', 'user_avg_Documentary',
       'user_avg_Drama', 'user_avg_Fantasy', 'user_avg_Film-Noir',
       'user_avg_Horror', 'user_avg_IMAX', 'user_avg_Musical',
       'user_avg_Mystery', 'user_avg_Romance', 'user_avg_Sci-Fi',
       'user_avg_Thriller', 'user_avg_War', 'user_avg_Western'],
      dtype='str')